# 👁️ VisionSpeak: IVIS — Google Colab Execution Notebook
### Student: Avasarala Sai Ganesh | Roll: 24M11MC007 | MCA | Aditya University | Guide: Dr. Bapuji Rao

**Notebook covers:**
1. Environment setup (Tesseract + Python packages)
2. OCR pipeline test
3. CNN Braille model definition + synthetic training
4. Braille recognition test
5. Text-to-Speech test
6. Accuracy metrics & visualisation
7. Launch Streamlit app via ngrok

> ⚡ **Set Runtime to GPU:** Runtime → Change runtime type → T4 GPU

## ✅ STEP 1 — Install System Packages (Tesseract)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-eng tesseract-ocr-hin tesseract-ocr-kan
!apt-get install -y -qq libgl1-mesa-glx libglib2.0-0
import subprocess
result = subprocess.run(['tesseract','--version'], capture_output=True, text=True)
print('Tesseract:', result.stdout.split('\n')[0])
print('✅ System packages installed')

## ✅ STEP 2 — Install Python Packages

In [ ]:
!pip install -q pytesseract pillow opencv-python-headless gtts streamlit pyngrok
print('✅ Python packages installed')

## ✅ STEP 3 — Imports & Verification

In [ ]:
import cv2, numpy as np, pytesseract, torch, torch.nn as nn
from PIL import Image, ImageDraw
from gtts import gTTS
from IPython.display import Audio, display
import matplotlib.pyplot as plt
import io, time, re

print('OpenCV :', cv2.__version__)
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
print('Tesseract langs:', pytesseract.get_languages())
print('✅ All imports successful')

## ✅ STEP 4 — Preprocessing Pipeline (OpenCV)

In [ ]:
def preprocess_image(pil_img, upscale=True):
    arr  = np.array(pil_img.convert('RGB'))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    den  = cv2.fastNlMeansDenoising(gray, h=10, templateWindowSize=7, searchWindowSize=21)
    if upscale:
        h, w = den.shape
        scale = max(1, min(4, 2400 // max(w, h, 1)))
        if scale > 1:
            den  = cv2.resize(den,  (w*scale, h*scale), interpolation=cv2.INTER_CUBIC)
            gray = cv2.resize(gray, (w*scale, h*scale), interpolation=cv2.INTER_CUBIC)
    clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    norm   = clahe.apply(den)
    binary = cv2.adaptiveThreshold(norm, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel)
    return binary, gray, norm

# Test with synthetic image
img = Image.new('RGB', (700, 120), 'white')
ImageDraw.Draw(img).text((20, 35), 'VisionSpeak IVIS - Testing OCR Pipeline', fill='black')
binary, gray, norm = preprocess_image(img)
fig, axes = plt.subplots(1, 3, figsize=(16, 3))
axes[0].imshow(img);            axes[0].set_title('Original');    axes[0].axis('off')
axes[1].imshow(norm, cmap='gray'); axes[1].set_title('CLAHE Normalised'); axes[1].axis('off')
axes[2].imshow(binary, cmap='gray'); axes[2].set_title('Binary (Thresholded)'); axes[2].axis('off')
plt.suptitle('OpenCV Preprocessing Pipeline', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('preprocessing.png', dpi=150); plt.show()
print('✅ Preprocessing pipeline verified')

## ✅ STEP 5 — Tesseract OCR Test

In [ ]:
def run_ocr(pil_img, lang='eng'):
    binary, _, _ = preprocess_image(pil_img, upscale=True)
    cfg  = f'--oem 3 --psm 6 -l {lang} -c preserve_interword_spaces=1'
    data = pytesseract.image_to_data(binary, config=cfg, output_type=pytesseract.Output.DICT)
    words, confs = [], []
    for i, w in enumerate(data['text']):
        c = int(data['conf'][i])
        if c > 25 and w.strip():
            words.append(w); confs.append(c)
    text = re.sub(r'\s{2,}', ' ', ' '.join(words)).strip()
    return {'text': text or 'No text', 'confidence': int(np.mean(confs)) if confs else 0}

# Test OCR
test_texts = [
    'VisionSpeak IVIS Project',
    'Avasarala Sai Ganesh - Roll 24M11MC007',
    'Braille Recognition using CNN and OCR',
    'Aditya University MCA Final Project 2025',
]
print(f'{'Ground Truth':<45} {'OCR Output':<45} {'Conf':>5}')
print('-' * 100)
for gt in test_texts:
    img = Image.new('RGB', (800, 80), 'white')
    ImageDraw.Draw(img).text((15, 20), gt, fill='black')
    res = run_ocr(img)
    print(f'{gt:<45} {res["text"]:<45} {res["confidence"]:>4}%')
print('\n✅ OCR test complete')

## ✅ STEP 6 — CNN Model Definition & Training

In [ ]:
# ── CNN Architecture (matches app.py exactly) ──────────────────────────────
class BrailleCNN(nn.Module):
    def __init__(self, num_classes=27):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32,32,3,padding=1), nn.ReLU(True),
            nn.MaxPool2d(2,2), nn.Dropout2d(0.25),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64,64,3,padding=1), nn.ReLU(True),
            nn.MaxPool2d(2,2), nn.Dropout2d(0.25),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(True), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = BrailleCNN(27).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model on   : {device}')
print(f'Parameters : {total_params:,}')
print(model)

In [ ]:
# ── Generate synthetic Braille training data ────────────────────────────────
DOT_TO_CHAR = {
    (): ' ', (1,):'a',(1,2):'b',(1,4):'c',(1,4,5):'d',(1,5):'e',
    (1,2,4):'f',(1,2,4,5):'g',(1,2,5):'h',(2,4):'i',(2,4,5):'j',
    (1,3):'k',(1,2,3):'l',(1,3,4):'m',(1,3,4,5):'n',(1,3,5):'o',
    (1,2,3,4):'p',(1,2,3,4,5):'q',(1,2,3,5):'r',(2,3,4):'s',(2,3,4,5):'t',
    (1,3,6):'u',(1,2,3,6):'v',(2,4,5,6):'w',(1,3,4,6):'x',(1,3,4,5,6):'y',(1,3,5,6):'z',
}
CHAR_TO_DOTS = {v:k for k,v in DOT_TO_CHAR.items()}
IDX_TO_CHAR  = [' '] + [chr(ord('a')+i) for i in range(26)]
CHAR_TO_IDX  = {c:i for i,c in enumerate(IDX_TO_CHAR)}

def make_cell_image(char, size=32, dot_r=3, noise_std=12):
    """Render a single Braille cell as a 32x32 grayscale numpy array."""
    img = np.ones((size, size), dtype=np.float32) * 255
    dots = CHAR_TO_DOTS.get(char, ())
    cx, cy = size//4, size//6
    pos = {1:(cx,cy), 2:(cx,cy*3), 3:(cx,cy*5),
           4:(cx*3,cy), 5:(cx*3,cy*3), 6:(cx*3,cy*5)}
    for dn, (x,y) in pos.items():
        color = 20 if dn in dots else 235
        cv2.circle(img, (x,y), dot_r, color, -1)
    # Add augmentation noise
    img += np.random.normal(0, noise_std, img.shape).astype(np.float32)
    img = np.clip(img, 0, 255)
    return img.astype(np.uint8)

# Build dataset: 300 samples per class
SAMPLES_PER_CLASS = 300
X, Y = [], []
for idx, char in enumerate(IDX_TO_CHAR):
    for _ in range(SAMPLES_PER_CLASS):
        cell = make_cell_image(char)
        X.append(cell)
        Y.append(idx)

X = np.array(X, dtype=np.float32) / 255.0
Y = np.array(Y, dtype=np.int64)
print(f'Dataset: {X.shape[0]} samples | {len(IDX_TO_CHAR)} classes')

# Sample visualisation
fig, axes = plt.subplots(3, 9, figsize=(18, 6))
for i, ax in enumerate(axes.flat):
    if i < len(IDX_TO_CHAR):
        ax.imshow(make_cell_image(IDX_TO_CHAR[i]), cmap='gray', vmin=0, vmax=255)
        ax.set_title(f"'{IDX_TO_CHAR[i]}'", fontsize=8)
    ax.axis('off')
plt.suptitle('Synthetic Braille Cell Training Samples', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('braille_samples.png', dpi=150); plt.show()
print('✅ Dataset generated')

In [ ]:
# ── Train the CNN ──────────────────────────────────────────────────────────
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch.optim as optim

Xt = torch.tensor(X).unsqueeze(1)   # (N, 1, 32, 32)
Yt = torch.tensor(Y)
dataset = TensorDataset(Xt, Yt)

train_size = int(0.85 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=2)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

EPOCHS = 20
train_accs, val_accs, train_losses = [], [], []

print(f'Training on {device} | {EPOCHS} epochs')
print(f'Train: {train_size} | Val: {val_size} samples')
print('-' * 55)

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out  = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (out.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    scheduler.step()

    model.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            vc += (model(xb).argmax(1) == yb).sum().item()
            vt += xb.size(0)

    tacc = correct/total*100; vacc = vc/vt*100; tloss = total_loss/total
    train_accs.append(tacc); val_accs.append(vacc); train_losses.append(tloss)
    print(f'Epoch {epoch:2d}/{EPOCHS} | Loss: {tloss:.4f} | Train Acc: {tacc:.1f}% | Val Acc: {vacc:.1f}%')

# Save checkpoint
torch.save(model.state_dict(), 'braille_cnn.pth')
print('\n✅ Model saved → braille_cnn.pth')
print(f'   Final Train Acc: {train_accs[-1]:.2f}%  |  Val Acc: {val_accs[-1]:.2f}%')

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
epochs = range(1, EPOCHS+1)

ax1.plot(epochs, train_accs, 'o-', color='#5eead4', label='Train Acc', lw=2)
ax1.plot(epochs, val_accs,   's--', color='#f59e0b',  label='Val Acc',   lw=2)
ax1.set_title('CNN Training & Validation Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy (%)'); ax1.legend()
ax1.set_ylim(0, 105); ax1.grid(alpha=0.3)

ax2.plot(epochs, train_losses, 'o-', color='#818cf8', lw=2)
ax2.set_title('Training Loss (CrossEntropy)', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved → training_curves.png')

## ✅ STEP 7 — Braille Recognition Test

In [ ]:
def create_braille_image(text, cell_size=80, dot_r=9):
    """Render a full Braille sentence as an image."""
    chars = [c for c in text.lower() if c in CHAR_TO_DOTS or c == ' ']
    w = len(chars)*cell_size + 40; h = cell_size + 40
    img = np.ones((h, w, 3), dtype=np.uint8)*255
    for i, ch in enumerate(chars):
        dots = CHAR_TO_DOTS.get(ch, ())
        bx = 20 + i*cell_size + cell_size//5
        by = 20
        pos = {1:(bx, by+5), 2:(bx, by+cell_size//3), 3:(bx, by+2*cell_size//3),
               4:(bx+cell_size//4, by+5), 5:(bx+cell_size//4, by+cell_size//3),
               6:(bx+cell_size//4, by+2*cell_size//3)}
        for dn, (x,y) in pos.items():
            color = (30,30,30) if dn in dots else (220,220,220)
            cv2.circle(img, (x,y), dot_r, color, -1)
    return img

# Test sentences
test_sentences = ['hello', 'vision', 'braille', 'mca']
print('Braille Recognition Test Results:')
print('-' * 50)
for sentence in test_sentences:
    braille_img = create_braille_image(sentence)
    pil_img = Image.fromarray(braille_img)

    # Run recognition pipeline
    binary, gray, _ = preprocess_image(pil_img, upscale=False)

    # Detect dots
    cnts, _ = cv2.findContours(binary.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    dots = []
    for c in cnts:
        area = cv2.contourArea(c)
        if 50 < area < 10000:
            perim = cv2.arcLength(c, True)
            if perim > 0 and 4*np.pi*area/(perim**2) > 0.35:
                (cx,cy),r = cv2.minEnclosingCircle(c)
                dots.append((int(cx),int(cy),max(1,int(r))))

    print(f'  Input: "{sentence}" | Dots detected: {len(dots)}')

# Visualise last sentence
plt.figure(figsize=(12, 3))
plt.imshow(create_braille_image('hello world'))
plt.title('Synthetic Braille: "hello world"', fontweight='bold')
plt.axis('off'); plt.savefig('braille_test.png', dpi=150); plt.show()
print('✅ Braille test complete')

## ✅ STEP 8 — Text-to-Speech Test

In [ ]:
from gtts import gTTS
from IPython.display import Audio

text = 'VisionSpeak IVIS: Intelligent Vision Interpretation System by Avasarala Sai Ganesh.'
buf = io.BytesIO()
gTTS(text=text, lang='en', slow=False).write_to_fp(buf)
buf.seek(0)
audio_bytes = buf.read()
print(f'✅ Audio generated: {len(audio_bytes)/1024:.1f} KB')
Audio(audio_bytes, autoplay=True)

## ✅ STEP 9 — Accuracy Metrics & Evaluation Report

In [ ]:
import difflib

def char_accuracy(pred, gt):
    if not gt: return 0.0
    return round(difflib.SequenceMatcher(None, pred.lower(), gt.lower()).ratio()*100, 2)

def word_accuracy(pred, gt):
    pw, gw = pred.lower().split(), gt.lower().split()
    if not gw: return 0.0
    return round(sum(p==g for p,g in zip(pw,gw))/len(gw)*100, 2)

# ── OCR test cases ──
ocr_cases = [
    'VisionSpeak IVIS Project', 'Aditya University MCA',
    'Braille Recognition System', 'Computer Vision AI',
    'Text to Speech Conversion', 'Dr Bapuji Rao Guide',
    'Optical Character Recognition', 'Deep Learning CNN Model',
    'Accessibility Assistive Tool', 'Speech Output Audio',
]
ocr_car, ocr_war = [], []
print('OCR Accuracy Evaluation:')
print(f'  {"Ground Truth":<35} {"CAR":>6}  {"WAR":>6}')
print('  ' + '-'*55)
for gt in ocr_cases:
    img = Image.new('RGB', (700, 70), 'white')
    ImageDraw.Draw(img).text((15, 18), gt, fill='black')
    res = run_ocr(img)
    car = char_accuracy(res['text'], gt)
    war = word_accuracy(res['text'], gt)
    ocr_car.append(car); ocr_war.append(war)
    print(f'  {gt:<35} {car:>5.1f}%  {war:>5.1f}%')

print(f'\n  Average CAR: {np.mean(ocr_car):.2f}%')
print(f'  Average WAR: {np.mean(ocr_war):.2f}%')

# ── CNN Braille accuracy ──
model.eval()
model_cpu = model.cpu()
correct = 0
for char in IDX_TO_CHAR:
    cell = make_cell_image(char, noise_std=5)
    t = torch.tensor(cell, dtype=torch.float32).unsqueeze(0).unsqueeze(0)/255.0
    with torch.no_grad():
        pred_idx = model_cpu(t).argmax(1).item()
    if IDX_TO_CHAR[pred_idx] == char:
        correct += 1
braille_acc = correct/len(IDX_TO_CHAR)*100
print(f'\n  CNN Braille Single-Char Accuracy: {braille_acc:.1f}% ({correct}/{len(IDX_TO_CHAR)} classes)')

In [ ]:
# ── Metrics charts ──
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Chart 1: OCR CAR
labels = [f'S{i+1}' for i in range(len(ocr_car))]
axes[0].bar(labels, ocr_car, color='#5eead4', edgecolor='#2a2a32', alpha=0.9)
axes[0].axhline(np.mean(ocr_car), color='red', ls='--', lw=2, label=f'Mean={np.mean(ocr_car):.1f}%')
axes[0].set_title('OCR Character Accuracy (CAR)', fontweight='bold')
axes[0].set_ylabel('Accuracy (%)'); axes[0].set_ylim(0,110); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Chart 2: OCR WAR
axes[1].bar(labels, ocr_war, color='#f59e0b', edgecolor='#2a2a32', alpha=0.9)
axes[1].axhline(np.mean(ocr_war), color='red', ls='--', lw=2, label=f'Mean={np.mean(ocr_war):.1f}%')
axes[1].set_title('OCR Word Accuracy (WAR)', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)'); axes[1].set_ylim(0,110); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

# Chart 3: Summary
metrics = ['OCR CAR', 'OCR WAR', 'CNN Braille']
values  = [np.mean(ocr_car), np.mean(ocr_war), braille_acc]
colors  = ['#5eead4', '#f59e0b', '#818cf8']
bars = axes[2].bar(metrics, values, color=colors, edgecolor='#2a2a32', alpha=0.9)
for bar, val in zip(bars, values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
                f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[2].set_title('Summary: All Metrics', fontweight='bold')
axes[2].set_ylabel('Accuracy (%)'); axes[2].set_ylim(0,115); axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('VisionSpeak: IVIS — Accuracy Evaluation Report', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('accuracy_report.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Accuracy report saved → accuracy_report.png')

## ✅ STEP 10 — Launch Streamlit App via ngrok

In [ ]:
# 1. Upload your app.py via the Files panel (left sidebar)
# 2. Get FREE ngrok token at https://ngrok.com → Dashboard → Authtoken
# 3. Replace the token below and run this cell

NGROK_TOKEN = 'PASTE_YOUR_NGROK_AUTHTOKEN_HERE'  # <-- REPLACE THIS

from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()   # kill any old tunnels

# Copy CNN checkpoint so app can load it
import shutil
if os.path.exists('braille_cnn.pth'):
    shutil.copy('braille_cnn.pth', '/content/braille_cnn.pth')

proc = subprocess.Popen(
    ['streamlit','run','app.py','--server.port=8501',
     '--server.headless=true','--server.fileWatcherType=none'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(6)
url = ngrok.connect(8501)

print('=' * 60)
print(f'  🚀 VisionSpeak: IVIS is LIVE at:')
print(f'  {url}')
print('=' * 60)
print('  Open the link above in any browser!')
print('  Share for live demo. Valid while Colab session is active.')

In [ ]:
# ── STOP the app ──
ngrok.kill()
proc.terminate()
print('✅ App stopped.')